In [10]:
from config.llm import llm
from langchain.agents import create_agent
from pydantic import BaseModel, Field
from langchain.tools import tool, ToolRuntime
from langchain.agents.middleware import dynamic_prompt, wrap_model_call, ModelRequest, ModelResponse, SummarizationMiddleware, HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from typing import Literal, Any

In [6]:
agent = create_agent(llm, system_prompt="you are a helpful assistant")
response = agent.invoke({'messages': '你好'})

response['messages'][-1].content

/Users/anotherbrick/Desktop/python/agent/.venv/lib/python3.11/site-packages/jwt/api_jwt.py:153: InsecureKeyLengthWarning: The HMAC key is 16 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(


'你好！很高兴见到你。有什么我可以帮助你的吗？'

带工具的agent

In [2]:
def get_weather(city: str) -> str:
  """get weather for a given city"""
  return f"it's always sunny in {city}"

In [7]:
tool_agent = create_agent(llm, system_prompt="you are a helpful assistant", tools=[get_weather])

response = tool_agent.invoke({
  "messages": [{
    "role": "user",
    "content": "what is the weather in sf"
  }]
})

response['messages'][-1]

/Users/anotherbrick/Desktop/python/agent/.venv/lib/python3.11/site-packages/jwt/api_jwt.py:153: InsecureKeyLengthWarning: The HMAC key is 16 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(
/Users/anotherbrick/Desktop/python/agent/.venv/lib/python3.11/site-packages/jwt/api_jwt.py:153: InsecureKeyLengthWarning: The HMAC key is 16 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(
/Users/anotherbrick/Desktop/python/agent/.venv/lib/python3.11/site-packages/jwt/api_jwt.py:153: InsecureKeyLengthWarning: The HMAC key is 16 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(
/Users/anotherbrick/Desktop/python/agent/.venv/lib/python3.11/site-packages/jwt/api_jwt.py:153: InsecureKeyLengthWarning: The HMAC key is 16 bytes long, which is below the 

AIMessage(content='\nI see that the weather function is returning a playful response about San Francisco being "always sunny." While San Francisco is known for its generally mild climate, it\'s actually not always sunny - the city experiences fog, especially in the summer months, and receives regular rainfall during the winter.\n\nIf you\'d like more specific weather information like current temperature, humidity, or detailed forecasts, you might want to check a dedicated weather service or app, as the function I have access to seems to be returning a simplified response.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 103, 'prompt_tokens': 235, 'total_tokens': 338}, 'model_name': 'glm-4', 'finish_reason': 'stop'}, id='lc_run--019ca4b5-9014-76c3-8845-04b7752e26dd-0', tool_calls=[], invalid_tool_calls=[])

In [21]:
class Context(BaseModel):
    authority: Literal["admin", "user"]

class CalcInfo(BaseModel):
    """Calculation information."""
    output: int = Field(description="The calculation result")

@tool
def math_add(runtime: ToolRuntime[Context, Any], a: int, b: int) -> int:
    """Add two numbers together"""
    authority = runtime.context.authority
    if authority != "admin":
        raise PermissionError("user dose not have permission to add numbers")
    return a + b

agent = create_agent(llm, system_prompt="you are a helpful assistant", tools=[math_add, get_weather], )

# tool_agent = create_agent(llm, system_prompt="you are a helpful assistant", tools=[math_add, get_weather], response_format=CalcInfo)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "请计算 8234783 + 94123832 = ?"}]},
    config={"configurable": {"thread_id": "1"}},
    context=Context(authority="admin"))

for message in response['messages']:
    message.pretty_print()

/Users/anotherbrick/Desktop/python/agent/.venv/lib/python3.11/site-packages/jwt/api_jwt.py:153: InsecureKeyLengthWarning: The HMAC key is 16 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(
/Users/anotherbrick/Desktop/python/agent/.venv/lib/python3.11/site-packages/jwt/api_jwt.py:153: InsecureKeyLengthWarning: The HMAC key is 16 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(


================================ Human Message =================================

请计算 8234783 + 94123832 = ?
================================== Ai Message ==================================


我来帮您计算这两个数字的和。
Tool Calls:
  math_add (call_-7848812936825921555)
 Call ID: call_-7848812936825921555
  Args:
    a: 8234783
    b: 94123832
================================= Tool Message =================================
Name: math_add

102358615
================================== Ai Message ==================================


8234783 + 94123832 = 102,358,615


In [3]:
agent = create_agent(llm, system_prompt="you are a helpful assistant", tools=[get_weather] )


for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode="updates",
):
    for step, data in chunk.items():
        print(f"step: {step}")
        print(f"content: {data['messages'][-1].content_blocks}")

step: model
content: [{'type': 'text', 'text': "\nI'll get the weather information for San Francisco (SF).\n"}, {'type': 'tool_call', 'id': 'call_-7848776824740899993', 'name': 'get_weather', 'args': {'city': 'SF'}}]
step: tools
content: [{'type': 'text', 'text': "it's always sunny in SF"}]
step: model
content: [{'type': 'text', 'text': '\nThe weather function returned a lighthearted response saying "it\'s always sunny in SF." This is a common phrase associated with San Francisco, though the actual weather can vary quite a bit throughout the year! \n\nSan Francisco typically has mild temperatures with cool, foggy mornings in the summer and occasional rain in the winter months. Would you like me to get more specific weather details for a particular date or time?'}]


In [11]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langchain.messages import HumanMessage, SystemMessage
from langgraph.prebuilt import ToolNode
from langchain_core.runnables import RunnableConfig

In [12]:
# 创建工具节点
tools = [get_weather]
tool_node = ToolNode(tools)

# 创建助手节点（修复核心错误）
def assistant(state: MessagesState, config: RunnableConfig):
    system_prompt = 'You are a helpful assistant that can check weather.'
    all_messages = [SystemMessage(system_prompt)] + state['messages']
    model = llm.bind_tools(tools=tools) 
    return {'messages': [model.invoke(all_messages)]}

# 创建条件边
def should_continue(state: MessagesState, config: RunnableConfig):
    messages = state['messages']
    last_message = messages[-1]
    if last_message.tool_calls:
        return 'continue'
    return 'end'

# 创建图
builder = StateGraph(MessagesState)

# 添加节点
builder.add_node('assistant', assistant)
builder.add_node('tool', tool_node)

# 添加边
builder.add_edge(START, 'assistant')

# 添加条件边
builder.add_conditional_edges(
    'assistant',
    should_continue,
    {
        'continue': 'tool',
        'end': END,
    },
)

# 添加边：调用工具节点后回到assistant
builder.add_edge('tool', 'assistant')

# 编译图
my_graph = builder.compile(name='my-graph')

# 调用图并输出结果
response = my_graph.invoke({'messages': [HumanMessage(content='上海天气怎么样？')]})
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

上海天气怎么样？
================================== Ai Message ==================================


我来帮您查询上海的天气情况。
Tool Calls:
  get_weather (call_-7848641481731472376)
 Call ID: call_-7848641481731472376
  Args:
    city: 上海
================================= Tool Message =================================
Name: get_weather

It's always sunny in 上海!
================================== Ai Message ==================================


根据查询结果，上海目前天气晴朗！阳光明媚，是个好天气。如果您需要更详细的天气信息（如温度、湿度、风力等），请告诉我，我可以为您进一步查询。


In [3]:
@tool
def add_numbers(a: float, b: float) -> float:
    """Add two numbers and return the sum."""
    return a + b

In [4]:
@tool
def calculate_bmi(weight_kg: float, height_m: float) -> float:
    """Calculate BMI given weight in kg and height in meters."""
    if height_m <= 0 or weight_kg <= 0:
        raise ValueError("height_m and weight_kg must be greater than 0.")
    return weight_kg / (height_m ** 2)

In [6]:
tool_agent = create_agent(
    model=llm,
    tools=[get_weather, add_numbers, calculate_bmi],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                # 无需触发人工审批
                "get_weather": False,
                # 需要审批，且允许approve,edit,reject三种审批类型
                "add_numbers": True,
                # 需要审批，允许approve,reject两种审批类型
                "calculate_bmi": {"allowed_decisions": ["approve", "reject"]},
            },
            description_prefix="Tool execution pending approval",
        ),
    ],
    checkpointer=InMemorySaver(),
    system_prompt="You are a helpful assistant",
)

In [8]:
import uuid
config = {'configurable': {'thread_id': str(uuid.uuid4())}}
result = tool_agent.invoke(
    {"messages": [{
        "role": "user",
        "content": "我身高180cm，体重180斤，我的BMI是多少"
        # "content": "what is the weather in sf"
    }]},
    config=config,
)

# result['messages'][-1].content
result.get('__interrupt__')

[Interrupt(value={'action_requests': [{'name': 'calculate_bmi', 'args': {'weight_kg': 90, 'height_m': 1.8}, 'description': "Tool execution pending approval\n\nTool: calculate_bmi\nArgs: {'weight_kg': 90, 'height_m': 1.8}"}], 'review_configs': [{'action_name': 'calculate_bmi', 'allowed_decisions': ['approve', 'reject']}]}, id='67e1da41da034a940f12e1a4a2b628ba')]

In [9]:
from langgraph.types import Command
result = tool_agent.invoke(
    Command(
        resume={"decisions": [{"type": "approve"}]}  # or "edit", "reject"
    ),
    config=config
)

result['messages'][-1].content

'\n根据您提供的信息，您的BMI是27.78。\n\n计算过程：\n- 身高：180cm = 1.8米\n- 体重：180斤 = 90公斤\n- BMI = 体重(kg) / 身高²(m²) = 90 / (1.8 × 1.8) = 27.78\n\n根据BMI标准，您的数值27.78属于"超重"范围（25-29.9）。建议您适当控制饮食、增加运动来维持健康的体重。'

In [13]:
@dynamic_prompt
def state_aware_prompt(request: ModelRequest) -> str:
    # request.messages is a shortcut for request.state["messages"]
    message_count = len(request.messages)

    base = "You are a helpful assistant."

    if message_count > 6:
        base += "\nThis is a long conversation - be extra concise."

    # 临时打印base看效果
    print(base)

    return base

agent = create_agent(
    model=llm,
    middleware=[state_aware_prompt]
)

result = agent.invoke(
    {"messages": [
        {"role": "user", "content": "广州今天的天气怎么样？"},
        {"role": "assistant", "content": "广州天气很好"},
        {"role": "user", "content": "吃点什么好呢"},
        {"role": "assistant", "content": "要不要吃香茅鳗鱼煲"},
        {"role": "user", "content": "香茅是什么"},
        {"role": "assistant", "content": "香茅又名柠檬草，常见于泰式冬阴功汤、越南烤肉"},
        {"role": "user", "content": "auv 那还等什么，咱吃去吧"},
    ]},
)

for message in result['messages']:
    message.pretty_print()

You are a helpful assistant.
This is a long conversation - be extra concise.
================================ Human Message =================================

广州今天的天气怎么样？
================================== Ai Message ==================================

广州天气很好
================================ Human Message =================================

吃点什么好呢
================================== Ai Message ==================================

要不要吃香茅鳗鱼煲
================================ Human Message =================================

香茅是什么
================================== Ai Message ==================================

香茅又名柠檬草，常见于泰式冬阴功汤、越南烤肉
================================ Human Message =================================

auv 那还等什么，咱吃去吧
================================== Ai Message ==================================


走！就去尝尝香茅鳗鱼煲
